In [ ]:
%load_ext autoreload
%autoreload 2


# Comparison with Observations of Nearby Galaxies (PHANGS)


## Introduction

Here, we utilize the PHANGS data release to recompute the gas weight and reproduce Figure 4 of Sun et al. 2023.

* Paper: https://ui.adsabs.harvard.edu/abs/2023ApJ...945L..19S
* Code: https://github.com/PhangsTeam/MegaTable
* Data: https://www.canfar.net/storage/vault/list/phangs/RELEASES/Sun_etal_2022


## Data Loading

We use the hexagon aperture (1.5 kpc) from the PHANGS megatable v4.0 (Sun et al. 2022).
`compute_prfm_inputs` adds the derived columns `Sigma_gas`, `Omega`, `qshear`, `H_star`, and their uncertainties.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from prfm import phangs
from prfm.phangs_plot import col_label, hist_plot, resolve_columns, scatter_plot

plt.rcParams["figure.dpi"] = 200

REPO_ROOT = Path("..")
DATA_DIR = REPO_ROOT / "data/phangs_megatable"
APERTURE = "hexagon"

tables = phangs.load_all(DATA_DIR, aperture=APERTURE)
t = phangs.vstack_tables(tables)
t = phangs.compute_prfm_inputs(t)

print(f"Aperture  : {APERTURE}")
print(f"Galaxies  : {len(tables)}")
print(f"Total rows: {len(t)}")


In [ ]:
mol  = np.isfinite(np.asarray(t["Sigma_mol"],  dtype=float))
atom = np.isfinite(np.asarray(t["Sigma_atom"], dtype=float))

both_valid = mol  &  atom
atom_only  = ~mol &  atom     # atom valid, mol NaN → mol filled with 0
mol_only   = mol  & ~atom     # mol valid, atom NaN → atom filled with 0
both_nan   = ~mol & ~atom     # both NaN → Sigma_gas = NaN

print(f"Total rows            : {len(t)}")
print(f"Both valid            : {both_valid.sum()}")
print(f"atom only (mol=0)     : {atom_only.sum()}")
print(f"mol only  (atom=0)    : {mol_only.sum()}")
print(f"Both NaN  (gas=NaN)   : {both_nan.sum()}")
print(f"Sigma_gas valid (new) : {np.isfinite(np.asarray(t['Sigma_gas'], dtype=float)).sum()}")


## Correlation Matrix

Scatter matrix for key gas and star-formation surface densities.
Diagonal panels show the PDF of each quantity; off-diagonal panels show pairwise scatter plots with error bars where available.


In [ ]:
from prfm.phangs_sampling import PHANGSSamplingDesigner, SamplingConfig

sampler = PHANGSSamplingDesigner(t, SamplingConfig())
fig, axes, selections = sampler.plot_correlation_matrix(aperture=APERTURE)
plt.show()


In [ ]:
# Select pixels in log Sigma_gas bands and overplot them on the correlation matrix.
from prfm.phangs_sampling import PHANGSSamplingDesigner, SamplingConfig

target_Sigma_gas_values = [10.0, 50.0]
delta_Sigma_gas = 0.15  # dex half-width around each target value

matrix_sampler = PHANGSSamplingDesigner(
    t,
    SamplingConfig(delta_sigma_gas=delta_Sigma_gas),
)
selected_pixels_by_target = matrix_sampler.select_sigma_gas_targets(
    target_Sigma_gas_values,
    delta_sigma_gas=delta_Sigma_gas,
)
for target, selected in selected_pixels_by_target.items():
    print(
        f"target Sigma_gas={target:g}, delta={delta_Sigma_gas:g} dex: "
        f"{len(selected)} selected pixels"
    )

fig, axes, selections = matrix_sampler.plot_correlation_matrix(
    targets=target_Sigma_gas_values,
    delta_sigma_gas=delta_Sigma_gas,
    aperture=APERTURE,
)
plt.show()


In [ ]:
# Observed-pixel Latin-hypercube sampling within one Sigma_gas slice.
SAMPLE_TARGET_SIGMA_GAS = 10.0
DELTA_SIGMA_GAS_DEX = 0.15
SAMPLE_RANDOM_SEED = 31415
LHS_SAMPLE_FIELDS = ["Sigma_star", "H_star", "Omega", "qshear"]
LHS_FAIRNESS_TOL_DEX = 0.15
LHS_SAMPLE_SIZE_GRID = [8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512]
LHS_N_REPEATS = 24

observed_sampler = PHANGSSamplingDesigner(
    t,
    SamplingConfig(
        target_sigma_gas=SAMPLE_TARGET_SIGMA_GAS,
        delta_sigma_gas=DELTA_SIGMA_GAS_DEX,
        fairness_tol_dex=LHS_FAIRNESS_TOL_DEX,
        random_seed=SAMPLE_RANDOM_SEED,
        n_repeats=LHS_N_REPEATS,
        sample_size_grid=LHS_SAMPLE_SIZE_GRID,
        primary_fields=LHS_SAMPLE_FIELDS,
        synthesis_method="observed_lhs",
    ),
)
reference_pixels = observed_sampler.select_reference_pixels()
required_n, sample_size_report = observed_sampler.estimate_required_samples(
    reference_pixels,
    method="observed_lhs",
    lhs_fields=LHS_SAMPLE_FIELDS,
    fairness_fields=LHS_SAMPLE_FIELDS,
)
print(
    f"Target Sigma_gas={SAMPLE_TARGET_SIGMA_GAS:g}, delta={DELTA_SIGMA_GAS_DEX:g} dex: "
    f"{len(reference_pixels)} eligible pixels before LHS filtering"
)
display(sample_size_report)
print(
    f"Required observed-pixel LHS sample size for {LHS_SAMPLE_FIELDS}: "
    f"n={required_n} using a {LHS_FAIRNESS_TOL_DEX:g} dex criterion."
)

lhs_sample = observed_sampler.sample(
    reference_pixels,
    required_n,
    method="observed_lhs",
    lhs_fields=LHS_SAMPLE_FIELDS,
)
primary_error, primary_fairness_report = observed_sampler.quantile_error_report(
    reference_pixels,
    lhs_sample,
    LHS_SAMPLE_FIELDS,
)
sfr_error, sfr_fairness_report = observed_sampler.quantile_error_report(
    reference_pixels,
    lhs_sample,
    observed_sampler.available_sfr_fields,
)

print("Primary-field fairness report:")
display(primary_fairness_report)
print("SFR fairness report:")
display(sfr_fairness_report)
print(
    "Does this observed LHS sample also fairly sample SFR space? "
    f"{'yes' if bool(sfr_fairness_report['fair'].all()) else 'no'}. "
    f"Worst SFR quantile mismatch: {sfr_error:.3f} dex."
)

fig_lhs_hist, axes_lhs_hist = observed_sampler.plot_distribution_overlay(
    reference_pixels,
    lhs_sample,
)
fig_lhs_pairs, axes_lhs_pairs = observed_sampler.plot_pair_overlay(
    reference_pixels,
    lhs_sample,
)
plt.show()

lhs_sample


In [ ]:
# Class-based synthetic/observed sampling workflow.
# Change these config values to test a different target, tolerance, or backend.
from prfm.phangs_sampling import PHANGSSamplingDesigner, SamplingConfig

SYNTH_TARGET_SIGMA_GAS = 10.0
SYNTH_DELTA_SIGMA_GAS_DEX = 0.15
SYNTH_FAIRNESS_TOL_DEX = 0.15
SYNTH_RANDOM_SEED = 27182
SYNTH_SFR_FIELD = "Sigma_SFR_Hacorr"

sampling_config = SamplingConfig(
    target_sigma_gas=SYNTH_TARGET_SIGMA_GAS,
    delta_sigma_gas=SYNTH_DELTA_SIGMA_GAS_DEX,
    fairness_tol_dex=SYNTH_FAIRNESS_TOL_DEX,
    random_seed=SYNTH_RANDOM_SEED,
    sfr_fields=SYNTH_SFR_FIELD,
    synthesis_method="kde_lhs",  # "kde_lhs", "expanded_kde_lhs", or "observed_lhs"
)

sampler = PHANGSSamplingDesigner(t, sampling_config)
reference_pixels = sampler.select_reference_pixels()
print(
    f"Target Sigma_gas={sampling_config.target_sigma_gas:g}, "
    f"delta={sampling_config.delta_sigma_gas:g} dex: "
    f"{len(reference_pixels)} reference pixels"
)

# Synthetic KDE-LHS fitted over gas + primary fields, with LHS coverage in primary fields.
required_primary_n, primary_size_report = sampler.estimate_required_samples(
    reference_pixels,
    method="kde_lhs",
    fit_fields=sampler.primary_fit_fields,
    lhs_fields=sampling_config.primary_fields,
    fairness_fields=sampling_config.primary_fields,
)
print("KDE-LHS sample-size report for primary-field fairness:")
display(primary_size_report)
if required_primary_n is None:
    required_primary_n = int(primary_size_report.iloc[-1]["n_samples"])
    print(
        "No tested primary sample size met the fairness criterion; "
        f"using the largest tested n={required_primary_n}."
    )
else:
    print(f"Required primary-field sample size: n={required_primary_n}")

synthetic_primary_sample = sampler.sample(
    reference_pixels,
    required_primary_n,
    method="kde_lhs",
    fit_fields=sampler.primary_fit_fields,
    lhs_fields=sampling_config.primary_fields,
)
primary_error, primary_report = sampler.quantile_error_report(
    reference_pixels,
    synthetic_primary_sample,
    sampling_config.primary_fields,
)
print("Primary-field fairness report:")
display(primary_report)

synthetic_primary_with_sfr = sampler.assign_sfr_from_neighbors(
    reference_pixels,
    synthetic_primary_sample,
    match_fields=sampler.primary_fit_fields,
    sfr_fields=sampler.available_sfr_fields,
    k_neighbors=16,
)
sfr_error, sfr_report = sampler.quantile_error_report_against_sample_field(
    reference_pixels,
    synthetic_primary_with_sfr,
    sampler.available_comparison_sfr_fields,
    SYNTH_SFR_FIELD,
)

print("SFR fairness report after nearest-neighbor SFR assignment:")
display(sfr_report)
print(
    "Does the synthetic primary KDE-LHS sample fairly sample SFR space after assignment? "
    f"{'yes' if bool(sfr_report['fair'].all()) else 'no'}. "
    f"Worst SFR quantile mismatch: {sfr_error:.3f} dex."
)

fig_kde_hist, axes_kde_hist = sampler.plot_distribution_overlay(
    reference_pixels,
    synthetic_primary_with_sfr,
    show_kde=True,
    sample_sfr_field=SYNTH_SFR_FIELD,
    comparison_sfr_fields=sampler.available_comparison_sfr_fields,
)
fig_kde_pairs, axes_kde_pairs = sampler.plot_pair_overlay(
    reference_pixels,
    synthetic_primary_with_sfr,
)
plt.show()

synthetic_primary_with_sfr


In [ ]:
# Expanded PHANGS-informed KDE-LHS prior for simulation design.
# This intentionally allows modest support beyond the observed PHANGS envelope.
expanded_config = SamplingConfig(
    target_sigma_gas=SYNTH_TARGET_SIGMA_GAS,
    delta_sigma_gas=SYNTH_DELTA_SIGMA_GAS_DEX,
    fairness_tol_dex=SYNTH_FAIRNESS_TOL_DEX,
    random_seed=SYNTH_RANDOM_SEED,
    sfr_fields=SYNTH_SFR_FIELD,
    synthesis_method="expanded_kde_lhs",
    expanded_prior_lower_dex={"Omega": 0.3, "H_star": 0.25, "qshear": 0.15},
    expanded_prior_upper_dex={},
    expanded_kde_bandwidth_factor=1.5,
)
expanded_sampler = PHANGSSamplingDesigner(t, expanded_config)
expanded_reference_pixels = expanded_sampler.select_reference_pixels()

expanded_sample = expanded_sampler.sample(
    expanded_reference_pixels,
    required_primary_n,
    method="expanded_kde_lhs",
    fit_fields=expanded_sampler.primary_fit_fields,
    lhs_fields=expanded_config.primary_fields,
)
expanded_error, expanded_report = expanded_sampler.quantile_error_report(
    expanded_reference_pixels,
    expanded_sample,
    expanded_config.primary_fields,
)
print("Expanded-prior design-field quantile report:")
display(expanded_report)
print(
    "Expanded-prior worst design-field quantile mismatch: "
    f"{expanded_error:.3f} dex. This mismatch is allowed to be larger "
    "when it reflects intentional prior expansion."
)

fig_expanded_hist, axes_expanded_hist = expanded_sampler.plot_distribution_overlay(
    expanded_reference_pixels,
    expanded_sample,
    show_kde=True,
)
fig_expanded_pairs, axes_expanded_pairs = expanded_sampler.plot_pair_overlay(
    expanded_reference_pixels,
    expanded_sample,
)
plt.show()

expanded_sample


In [ ]:

# Joint KDE-LHS fitted over gas + primary + SFR fields. This gives synthetic SFR values.
required_joint_n, joint_size_report = sampler.estimate_required_samples(
    reference_pixels,
    method="kde_lhs",
    fit_fields=sampler.joint_fit_fields,
    lhs_fields=sampling_config.primary_fields,
    fairness_fields=sampler.joint_fit_fields,
)
print("KDE-LHS sample-size report requiring gas + primary + SFR fairness:")
display(joint_size_report)
if required_joint_n is None:
    required_joint_n = int(joint_size_report.iloc[-1]["n_samples"])
    print(
        "No tested joint sample size met the fairness criterion; "
        f"using the largest tested n={required_joint_n}."
    )
else:
    print(f"Required joint sample size: n={required_joint_n}")

synthetic_joint_sample = sampler.sample(
    reference_pixels,
    required_joint_n,
    method="kde_lhs",
    fit_fields=sampler.joint_fit_fields,
    lhs_fields=sampling_config.primary_fields,
)
joint_error, joint_report = sampler.quantile_error_report(
    reference_pixels,
    synthetic_joint_sample,
    sampler.joint_fit_fields,
)
sfr_error, sfr_report = sampler.quantile_error_report_against_sample_field(
    reference_pixels,
    synthetic_joint_sample,
    sampler.available_comparison_sfr_fields,
    SYNTH_SFR_FIELD,
)
print("Joint-field fairness report:")
display(joint_report)
print("SFR fairness report:")
display(sfr_report)
print(
    "Does the synthetic joint KDE-LHS sample fairly sample SFR space? "
    f"{'yes' if bool(sfr_report['fair'].all()) else 'no'}. "
    f"Worst SFR quantile mismatch: {sfr_error:.3f} dex."
)

fig_kde_hist, axes_kde_hist = sampler.plot_distribution_overlay(
    reference_pixels,
    synthetic_joint_sample,
    show_kde=True,
    sample_sfr_field=SYNTH_SFR_FIELD,
    comparison_sfr_fields=sampler.available_comparison_sfr_fields,
)
fig_kde_pairs, axes_kde_pairs = sampler.plot_pair_overlay(
    reference_pixels,
    synthetic_joint_sample,
)
plt.show()

synthetic_joint_sample
